# Feature Engineering and Recommendation

## Loading the Data and doing the check\-ups

In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
tracks = pd.read_csv("data/SpotGenTrack/Data Sources/spotify_tracks.csv")
artists = pd.read_csv("data/SpotGenTrack/Data Sources/spotify_artists.csv")
albums = pd.read_csv("data/SpotGenTrack/Data Sources/spotify_albums.csv")
audio = pd.read_csv("data/SpotGenTrack/Features Extracted/low_level_audio_features.csv")
lyrics = pd.read_csv("data/SpotGenTrack/Features Extracted/lyrics_features.csv")

In [5]:
datasets = {
    "tracks": tracks,
    "artists": artists,
    "albums": albums,
    "audio": audio,
    "lyrics": lyrics
}

for name, df in datasets.items():
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)
        print(f"Removed Unnamed: 0 from {name}")

Removed Unnamed: 0 from tracks
Removed Unnamed: 0 from artists
Removed Unnamed: 0 from albums
Removed Unnamed: 0 from audio
Removed Unnamed: 0 from lyrics


In [6]:
for name, df in datasets.items():
    print(name, df.shape)

tracks (101939, 31)
artists (56129, 8)
albums (75511, 15)
audio (101909, 208)
lyrics (94954, 7)


In [7]:
base_features = [
    "id",
    "name",
    "artists_id",
    "album_id",
    "popularity",
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness",
    "loudness",
    "tempo"
]

tracks_base = tracks[base_features].copy()

tracks_base.head()

,id,name,artists_id,album_id,popularity,danceability,energy,valence,acousticness,instrumentalness,speechiness,liveness,loudness,tempo
0,5qljLQuKnNJf4F4vfxQB0V,Blood,['3mxJuHRn2ZWD5OofvJtDZY'],0D3QufeCudpQANOR7luqdr,28.0,0.698,0.606,0.6220,0.294,0.000003,0.0262,0.151,-7.447,115.018
1,3VAX2MJdmdqARLSU5hPMpm,The Ugly Duckling,['4xWMewm6CYMstu0sPgd9jJ'],1bcqsH5UyTBzmh9YizdsBE,31.0,0.719,0.308,0.5890,0.863,0.000000,0.9220,0.253,-10.340,115.075
2,1L3YAhsEMrGVvCgDXj2TYn,Jimmy Launches His Own Range Of Greetings Cards,['3hYaK5FF3YAglCj5HZgBnP'],4tKijjmxGClg4JOLAyo2qE,31.0,0.466,0.931,0.0850,0.750,0.000000,0.9440,0.938,-13.605,79.565
3,6aCe9zzoZmCojX7bbgKKtf,The Crime At Pickets Mill,['2KQsUB9DRBcJk17JWX1eXD'],6FeJF5r8roonnKraJxr4oB,14.0,0.719,0.126,0.5330,0.763,0.000000,0.9380,0.113,-20.254,112.822
4,1Vo802A38tPFHmje1h91um,Setup + Punchline = Joke,['3hYaK5FF3YAglCj5HZgBnP'],4tKijjmxGClg4JOLAyo2qE,32.0,0.460,0.942,0.0906,0.770,0.000000,0.9430,0.917,-13.749,81.260


In [8]:
tracks_base.isnull().sum()

id                  0
name                0
artists_id          0
album_id            0
popularity          0
danceability        0
energy              0
valence             0
acousticness        0
instrumentalness    0
speechiness         0
liveness            0
loudness            0
tempo               0
dtype: int64

In [9]:
recommendation_features = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness",
    "loudness",
    "tempo"
]

In [10]:
scaler = StandardScaler()

tracks_scaled = scaler.fit_transform(tracks_base[recommendation_features])

tracks_scaled.shape

(101939, 9)

After reloading the datasets, the automatically generated index column \(\`Unnamed: 0\`\) was removed again to ensure that this notebook can run independently from the previous EDA notebook\. The resulting datasets remain large and complete enough for recommendation modeling, with the main tracks dataset containing 101,939 records and the selected baseline feature matrix containing 101,939 tracks with 9 numerical audio features\. This baseline feature set is intentionally kept simple and interpretable at this stage, so it can serve as a first version of the content\-based recommendation system before adding more complex features such as lyrics, genres, or low\-level audio descriptors\.

## Dataset merging requirements

In [11]:
# Check important ID columns before merging

print("Tracks IDs:", tracks["id"].nunique())
print("Audio track IDs:", audio["track_id"].nunique())
print("Lyrics track IDs:", lyrics["track_id"].nunique())
print("Artists track IDs:", artists["track_id"].nunique())
print("Albums track IDs:", albums["track_id"].nunique())

Tracks IDs: 101939
Audio track IDs: 101909
Lyrics track IDs: 94954
Artists track IDs: 44895
Albums track IDs: 75511


In [12]:
# Check how many tracks can be matched with each additional dataset

track_ids = set(tracks["id"])

audio_matches = len(track_ids.intersection(set(audio["track_id"])))
lyrics_matches = len(track_ids.intersection(set(lyrics["track_id"])))
artist_matches = len(track_ids.intersection(set(artists["track_id"])))
album_matches = len(track_ids.intersection(set(albums["track_id"])))

print(f"Tracks matched with audio features: {audio_matches}")
print(f"Tracks matched with lyrics features: {lyrics_matches}")
print(f"Tracks matched with artists data: {artist_matches}")
print(f"Tracks matched with albums data: {album_matches}")

Tracks matched with audio features: 101909
Tracks matched with lyrics features: 94954
Tracks matched with artists data: 44892
Tracks matched with albums data: 75502


In [13]:
# Convert matches into percentages for easier interpretation

total_tracks = len(tracks)

match_summary = pd.DataFrame({
    "Dataset": ["Audio", "Lyrics", "Artists", "Albums"],
    "Matched Tracks": [audio_matches, lyrics_matches, artist_matches, album_matches],
    "Match Percentage": [
        audio_matches / total_tracks * 100,
        lyrics_matches / total_tracks * 100,
        artist_matches / total_tracks * 100,
        album_matches / total_tracks * 100
    ]
})

match_summary["Match Percentage"] = match_summary["Match Percentage"].round(2)

match_summary

,Dataset,Matched Tracks,Match Percentage
0,Audio,101909,99.97
1,Lyrics,94954,93.15
2,Artists,44892,44.04
3,Albums,75502,74.07


The dataset overlap analysis shows that the different Spotify\-related datasets can be merged with varying degrees of completeness\. The audio features dataset has an almost perfect overlap with the tracks dataset \(99\.97%\), making it highly suitable for a baseline content\-based recommendation system\. The lyrics dataset also shows strong coverage \(93\.15%\), allowing future integration of lyric\-based features\. Album metadata is available for most tracks \(74\.07%\), while the artists dataset currently shows lower direct overlap \(44\.04%\), likely due to the presence of multiple artists per track and list\-based artist identifiers\. Overall, the results indicate that the datasets are sufficiently compatible for recommendation modeling, while also highlighting areas that may require additional preprocessing in later stages\.

## Baseline Recommender

In [14]:
# Scale recommendation features

scaler = StandardScaler()
tracks_scaled = scaler.fit_transform(tracks_base[recommendation_features])

In [15]:
# Fit nearest-neighbor model

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

knn_model = NearestNeighbors(
    n_neighbors=6,
    metric="cosine",
    algorithm="brute"
)

knn_model.fit(tracks_scaled)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)

In [16]:
# Recommendation function using KNN

def recommend_tracks_knn(track_name, top_n=5):
    matches = tracks_base[tracks_base["name"].str.lower() == track_name.lower()]
    
    if matches.empty:
        return f"Track '{track_name}' not found."
    
    track_idx = matches.index[0]
    track_vector = tracks_scaled[track_idx].reshape(1, -1)
    
    distances, indices = knn_model.kneighbors(track_vector, n_neighbors=top_n + 1)
    
    recommended_indices = indices[0][1:]
    
    recommendations = tracks_base.iloc[recommended_indices][
        ["name", "popularity", "danceability", "energy", "valence", "tempo"]
    ].copy()
    
    recommendations["similarity"] = 1 - distances[0][1:]
    
    return recommendations

In [24]:
recommend_tracks_knn("Billie Jean")

,name,popularity,danceability,energy,valence,tempo,similarity
101367,Billie Jean,84.0,0.920,0.654,0.847,117.046,0.999781
97241,Para Enamorarte,64.0,0.817,0.676,0.796,115.999,0.990172
81305,Desde o Início - Beowülf Remix,32.0,0.871,0.647,0.821,124.017,0.989853
9748,Who's Up?,43.0,0.915,0.622,0.856,118.076,0.989683
101835,Checklist (with Calvin Harris) (feat. WizKid),64.0,0.890,0.624,0.762,116.046,0.988913


A first baseline recommendation model was implemented using K\-Nearest Neighbors with cosine distance on standardized Spotify audio features\. A full pairwise cosine similarity matrix was not feasible in the Deepnote Basic environment because it would require comparing over 100,000 tracks with each other, leading to excessive memory usage\. Therefore, a KNN\-based approach was used instead, as it retrieves the nearest tracks without storing the full similarity matrix\. The resulting recommendations are based only on numerical audio similarity, such as danceability, energy, valence, acousticness, loudness, and tempo\. While this confirms that the technical recommendation pipeline works, the current output should be interpreted as a simple baseline rather than a final recommender\. Future improvements should include additional context such as genre, artist information, lyrics, popularity, or low\-level audio features to improve recommendation quality\.

I added the first baseline recommender\. First tried full cosine similarity, but that creates a huge 100k x 100k matrix and Deepnote does not have enough memory for that\. So I switched to a KNN approach using cosine distance, which is more scalable\. The recommender works technically and returns songs with similar audio feature values, but it is still quite basic because it only uses Spotify audio features for now\. The next step should be improving it with more context, probably genre/artist info, lyrics, or low\-level audio features\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=f51f68fc-16af-4ff9-b4f3-009bd2125c28' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>